# Feasibility Analysis: Diagnosis-Procedure Prediction

This notebook evaluates whether diagnosis patterns, DRG, severity indicators, and encounter characteristics contain enough predictive signal to estimate whether selected procedure CCSR categories occur during an inpatient admission.

The purpose of this notebook is not to select final production models. Instead, it serves as a feasibility check to determine whether machine learning is a viable foundation for the revenue integrity review framework.

In [19]:
import os
from datetime import datetime
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

### Data directory setup
This notebook assumes that you can access the **Quantic Capstone** Google Drive set up for this project as a local path on your personal computer.<br>
Before running the notebook, set `base_directory` in the cell immediately below to the folder location on your machine.

If your local path is different, update `base_directory` so it points to your own accessible **Quantic Capstone** directory before running the next cell.

In [20]:
base_directory = "/Users/scott/Library/CloudStorage/GoogleDrive-clt.av8or@gmail.com/My Drive/Quantic Capstone"

In [21]:
# Working directory
working_dir=base_directory + '/Models/feasibility_analysis/'

In [22]:
# Read input file
input_file_nm='feasibility_model_input.csv'
df = pd.read_csv(working_dir + input_file_nm)

In [23]:
 # Inspect dataframe structure, column types, and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 560311 entries, 0 to 560310
Data columns (total 40 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   admit_id              560311 non-null  int64 
 1   drg_cd                560311 non-null  int64 
 2   age                   560311 non-null  int64 
 3   gender                560280 non-null  object
 4   rom_score             560311 non-null  int64 
 5   soi_score             560311 non-null  int64 
 6   length_of_stay        560311 non-null  int64 
 7   total_charge          560311 non-null  int64 
 8   diagnosis_count       560311 non-null  int64 
 9   distinct_ccsr_count   560311 non-null  int64 
 10  secondary_ccsr_count  560311 non-null  int64 
 11  sec_ccsr_bld003       560311 non-null  int64 
 12  sec_ccsr_cir007       560311 non-null  int64 
 13  sec_ccsr_cir008       560311 non-null  int64 
 14  sec_ccsr_cir011       560311 non-null  int64 
 15  sec_ccsr_cir017  

In [24]:
# Define target columns
target_cols = [
'target_prccsr_car024',
'target_prccsr_esa001',
'target_prccsr_esa004',
'target_prccsr_gis009',
'target_prccsr_img008',
'target_prccsr_mst019',
'target_prccsr_res005'
]

## Target Procedure Categories
Seven procedure CCSR targets were evaluated during feasibility testing. The notebook is run once per target by updating the `target` variable below. Each target represents a binary indicator showing whether that procedure category was present for the admission.

In [25]:
# Execute this entire Jupyter notebook once for each of the 7 targets below,
# noting the model scores for each target
target='target_prccsr_car024'
#target='target_prccsr_esa001'
#target='target_prccsr_esa004'
#target='target_prccsr_gis009'
#target='target_prccsr_img008'
#target='target_prccsr_mst019'
#target='target_prccsr_res005'

## Feature Preparation
The model input contains admission-level features including DRG, age, severity scores, diagnosis counts, and selected secondary diagnosis CCSR indicators. Target columns, identifiers, gender text, and financial fields are excluded from the feature matrix to avoid leakage or non-modeling fields.

In [26]:
# Encode Gender
df['gender_female'] = (df['gender'] == 'F').astype(int)

# Convert DRG to string
df['drg_cd'] = df['drg_cd'].astype(str)

In [27]:
# Standardize numeric features so Logistic Regression is not influenced by feature scale
from sklearn.preprocessing import StandardScaler
numeric_cols=[
    'age',
    'rom_score',
    'soi_score',
    'length_of_stay',
    'diagnosis_count',
    'distinct_ccsr_count',
    'secondary_ccsr_count']

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])


In [28]:
# Define X and y
X=df.drop(columns=target_cols + ['admit_id','gender','total_charge'])
y=df[target]

In [29]:
# One-hot encode DRG
X=pd.get_dummies(X,columns=['drg_cd'],drop_first=True)

In [30]:
# Train/Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [31]:
# Model 1: Logistic Regression
from sklearn.linear_model import LogisticRegression
log_model = LogisticRegression(
    max_iter=5000,
    random_state=42,
    solver='lbfgs',
     )
log_model.fit(X_train, y_train)
y_prob_log = log_model.predict_proba(X_test)[:,1]

In [32]:
# Model 2: Random Forest baseline to evaluate nonlinear relationships
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [33]:
# Evaluate models
# What we are looking for: AUC > 0.75
from sklearn.metrics import roc_auc_score

auc_log = roc_auc_score(y_test, y_prob_log)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print(f"Target: {target}")
print(f"Logistic AUC: {auc_log:.3f}")
print(f"Random Forest AUC: {auc_rf:.3f}")

Target: target_prccsr_car024
Logistic AUC: 0.841
Random Forest AUC: 0.853


The notebook was executed separately for each candidate procedure target.<br>
The resulting ROC-AUC scores are summarized below.<br><br>
Target: target_prccsr_car024<br>
Logistic AUC: 0.841<br>
Random Forest AUC: 0.853<br><br>

Target: target_prccsr_esa001<br>
Logistic AUC: 0.835<br>
Random Forest AUC: 0.824<br><br>

Target: target_prccsr_esa004<br>
Logistic AUC: 0.835<br>
Random Forest AUC: 0.824<br><br>

Target: target_prccsr_gis009<br>
Logistic AUC: 0.980<br>
Random Forest AUC: 0.976<br><br>

Target: target_prccsr_img008<br>
Logistic AUC: 0.703<br>
Random Forest AUC: 0.683<br><br>

Target: target_prccsr_mst019<br>
Logistic AUC: 0.924<br>
Random Forest AUC: 0.897<br><br>

Target: target_prccsr_res005<br>
Logistic AUC: 0.799<br>
Random Forest AUC: 0.795<br>

## Interpretation
The feasibility results support continued model development. Most candidate procedure targets achieved ROC-AUC scores above 0.80, indicating that diagnosis patterns, DRG, SOI, ROM, and encounter-level characteristics contain meaningful predictive signal.
<br>
In practical terms, the results suggest that:
- Certain diagnosis patterns are strongly associated with specific procedure categories.
- Severity measures add useful clinical context.
- DRG helps anchor comparisons among clinically similar admissions.
<br>
Based on these findings, the project proceeded to full model development and threshold optimization.

In [35]:
# Precision and Recall
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_test, y_prob_rf)

print(f"Max Recall: {recall.max():.3f}")
print(f"Max Precision: {precision.max():.3f}")

Max Recall: 1.000
Max Precision: 1.000


## Precision and Recall Diagnostic
Max Recall and Max Precision are the same for all targets:<br>
Max Recall: 1.000<br>
Max Precision: 1.000
<br>
Maximum precision and maximum recall are not meaningful as standalone feasibility metrics because they occur at extreme threshold values. <br>
However, the result confirms that threshold optimization will be required in later modeling phases to balance recall, precision, and review queue size.